# Notebook 01c · Milan, AlphaEarth embedding, GHS-BUILT-S labels

Predicts impervious surface density (%) for Milan from the 64 AlphaEarth bands, trained on
GHS-BUILT-S. All 3,500 points of the feature table (`data/sample_features/`,
written by notebook 00) are used; there is no train/test split.

**The random forest is tuned and built in Earth Engine** (`gee_models.py`), so
the map is the tuned model by construction:

1. Tuning: 40 settings x 3 block sizes (500 / 1000 / 2000 m), 5 spatial-block
   folds each, scored in Earth Engine; the lowest mean fold RMSE wins.
2. Spatial CV metrics: out-of-fold predictions at the chosen block size.
3. Final model: the chosen setting trained on all points; its record
   (`model_rf.json`: points, settings, seed, CV metrics, importances) is kept
   beside the map and rebuilds the identical forest (`gee_models.load_model`).
4. The map is exported and downloaded automatically (`gee_export.py`).

Accuracy against the ground is notebook 05 (EarthLabel); notebook 04 compares
the whole map with its training product.

---
## Cell 1 · Imports & configuration

In [1]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

NameError: name 'ee' is not defined

In [1]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

In [2]:
import os, pyproj
_proj = pyproj.datadir.get_data_dir()
os.environ['PROJ_LIB']  = _proj
os.environ['PROJ_DATA'] = _proj

In [3]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee
from gee_export import start_export, wait_and_download
import sample_features as sf
import gee_models as gm

init_gee()

import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings, os
import matplotlib.ticker as mticker
import rioxarray as rxr
warnings.filterwarnings('ignore')
# ── Configuration ──────────────────────────────────────────────────────────────
from gee_init import get_project
GEE_PROJECT    = get_project()   # from .env
AEF_YEAR       = 2018
RANDOM_STATE   = 42
N_JOBS         = -1       # all cores; with a fixed random_state RF results do not depend on n_jobs
N_FOLDS        = 5        # CV folds within training set
INNER_CV       = 5        # inner folds for HP tuning (can be changed, for example 3 or 5)
BUFFER_M       = 250      # buffer between train/test and between CV folds
SPLIT_BLOCK_M  = 1000     # block size for train/test split
BLOCK_SIZES    = [500, 1000, 2000]   # block sizes tested for HP tuning CV
TEST_FRAC      = 0.30     # target test fraction
N_ITER_RF      = 40
N_CLASSES      = 7
# ── Paths ──────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = './output/milan/ghsl/embedding'
RF_RASTER_PATH  = f'{OUTPUT_DIR}/GHSL_predicted_RF_spatialCV2_Milan.tif'
os.makedirs(OUTPUT_DIR, exist_ok=True)
# ── Plot style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
})
CLASS_COLORS = ['#2166ac','#74add1','#abd9e9','#fee090','#f46d43','#d73027','#a50026']
CLASS_LABELS = ['C0 (0%)','C1 (1-20%)','C2 (21-40%)','C3 (41-60%)','C4 (61-80%)','C5 (81-99%)','C6 (100%)']
MODEL_COLORS = {'RF': '#2ecc71'}
CV_LABELS    = ['Random'] + [f'{bs}m' for bs in BLOCK_SIZES]
print('Setup OK')
print(f'Split block: {SPLIT_BLOCK_M}m | Buffer: {BUFFER_M}m | Target test: {TEST_FRAC*100:.0f}%')
print(f'CV folds: {N_FOLDS} outer / {INNER_CV} inner | Block sizes: {BLOCK_SIZES} m')

Setup OK
Split block: 1000m | Buffer: 250m | Target test: 30%
CV folds: 5 outer / 5 inner | Block sizes: [500, 1000, 2000] m


---
## Cell 2 · Load GEE assets & all 3500 points

In [4]:
# ── GEE ───────────────────────────────────────────────────────────────────────
aoi      = ee.FeatureCollection(f'projects/{GEE_PROJECT}/assets/milano_aoi')
aoi_geom = aoi.geometry()

non_water_mask = (ee.ImageCollection('ESA/WorldCover/v200')
                  .filter(ee.Filter.bounds(aoi_geom))
                  .mosaic().clip(aoi_geom)).neq(80)

start_date = ee.Date.fromYMD(AEF_YEAR, 1, 1)
aef_col    = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
              .filter(ee.Filter.date(start_date, start_date.advance(1, 'year')))
              .filter(ee.Filter.bounds(aoi_geom)))
aef_proj   = aef_col.first().projection()
aef_img    = (aef_col.mosaic()
              .setDefaultProjection(aef_proj)
              .clip(aoi_geom)
              .updateMask(non_water_mask))

band_names = aef_img.bandNames().getInfo()
print(f'AEF bands: {len(band_names)}')

# ── Points, labels, split and embedding bands (notebook 00) ───────────────────
gdf_all, feat_meta = sf.load_features('GHSL', 'Milan', 'embedding')
assert feat_meta['band_names'] == band_names, (
    'Feature table bands differ from the AlphaEarth image; re-run notebook 00.')

gdf_utm = gdf_all.to_crs('EPSG:32632')
coords_all = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])

X_all = gdf_all[band_names].values.astype(np.float32)
y_all = gdf_all['GHSL'].values.astype(np.float32)

print(f'Total points: {len(gdf_all)}')
print(f'X shape: {X_all.shape} | y range: {y_all.min():.0f}-{y_all.max():.0f}%')
print(f'Class distribution:')
for c in range(N_CLASSES):
    n = int((gdf_all['GHSL_class']==c).sum())
    print(f'  {CLASS_LABELS[c]}: {n}')


AEF bands: 64
Total points: 3500
X shape: (3500, 64) | y range: 0-100%
Class distribution:
  C0 (0%): 500
  C1 (1-20%): 500
  C2 (21-40%): 500
  C3 (41-60%): 500
  C4 (61-80%): 500
  C5 (81-99%): 500
  C6 (100%): 500


---
## Cell 3 · Tune in Earth Engine

Every setting is trained and scored on the 5 spatial-block folds of each block
size by Earth Engine (one request per setting and block, 8 in parallel). The
full table is saved; the chosen setting is the lowest mean fold RMSE.

In [ ]:
LABEL, SOURCE, PREDICTOR = 'GHSL', 'GHSL', 'embedding'
MODEL_PATH = f'{OUTPUT_DIR}/model_rf.json'

fc = gm.points_fc(gdf_all, band_names, LABEL, gm.make_folds(coords_all))
tune_rows, best = gm.tune(fc, band_names, LABEL)
pd.DataFrame(tune_rows).to_csv(f'{OUTPUT_DIR}/hyperparameter_tuning.csv', index=False)
print(f'Saved {OUTPUT_DIR}/hyperparameter_tuning.csv ({len(tune_rows)} rows)')

---
## Cell 4 · Spatial CV metrics (out-of-fold)

Each point predicted by the chosen setting trained without its spatial block
(chosen block size). RMSE, MAE, R² and bias (observed minus predicted).

In [ ]:
oof_pred, cv = gm.oof_predictions(fc, band_names, LABEL, best)
pd.DataFrame({'pid': gdf_all['pid'].values, LABEL: y_all, 'pred_oof': oof_pred}).to_csv(
    f'{OUTPUT_DIR}/oof_predictions.csv', index=False)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_all, oof_pred, s=6, alpha=0.35, linewidths=0)
ax.plot([0, 100], [0, 100], 'k--', lw=1)
ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.set_aspect('equal')
ax.set_xlabel(f'{LABEL} (%)'); ax.set_ylabel('Out-of-fold prediction (%)')
ax.text(0.04, 0.96, f"RMSE {cv['rmse']:.2f}\nMAE {cv['mae']:.2f}\nR² {cv['r2']:.3f}\n"
        f"bias {cv['bias']:+.2f}\nblock {cv['block']} m", transform=ax.transAxes, va='top', fontsize=9,
        bbox=dict(facecolor='white', alpha=0.8, edgecolor='grey'))
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig_oof_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Cell 5 · Final model and map

The chosen setting trained on all points, its record saved (`model_rf.json`),
and the full-resolution map exported and downloaded to `RF_RASTER_PATH`.

In [ ]:
gee_rf = gm.train_final(fc, band_names, LABEL, best)
record = gm.save_model(gee_rf, gm.model_record(best, tune_rows, cv, SOURCE, 'Milan', PREDICTOR,
                                               band_names, LABEL, len(gdf_all)), MODEL_PATH)

prediction_rf = (aef_img.classify(gee_rf).rename('GHSL_predicted')
                 .clip(aoi_geom).updateMask(non_water_mask))
wait_and_download([start_export(prediction_rf, RF_RASTER_PATH,
                                region=aoi_geom, crs='EPSG:32632')])

---
## Cell 21 · Figure E — Raster comparison map

In [ ]:
# Figure E -- Raster comparison
DISPLAY_SCALE  = 2
GHSL_OBS_PATH  = 'data/GHSL_2018_Milan_UTM32N.tif'
GHSL_PRED_PATH = RF_RASTER_PATH

ghsl_obs  = rxr.open_rasterio(GHSL_OBS_PATH,  masked=False).squeeze('band', drop=True)[::DISPLAY_SCALE, ::DISPLAY_SCALE]
ghsl_pred = rxr.open_rasterio(GHSL_PRED_PATH, masked=False).squeeze('band', drop=True)[::DISPLAY_SCALE, ::DISPLAY_SCALE]
if ghsl_pred.rio.crs != ghsl_obs.rio.crs or ghsl_pred.shape != ghsl_obs.shape:
    ghsl_pred = ghsl_pred.rio.reproject_match(ghsl_obs)
ghsl_diff = ghsl_pred - ghsl_obs
ghsl_cmap = ListedColormap(['#1a9641','#a6d96a','#ffffbf','#fdae61','#d7191c'])

def imshow_aspect(ax, da, **kw):
    b  = da.rio.bounds()
    im = ax.imshow(da.values, extent=(b[0], b[2], b[1], b[3]), **kw)
    ax.set_aspect('equal')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=4))
    ax.tick_params(axis='both', labelsize=8)
    ax.set_xlabel('Easting [m]', fontsize=9); ax.set_ylabel('Northing [m]', fontsize=9)
    return im

fig, axes_e = plt.subplots(2, 2, figsize=(15, 12), gridspec_kw={'wspace': 0.25, 'hspace': 0.04})
im1 = imshow_aspect(axes_e[0,0], ghsl_obs,  cmap=ghsl_cmap, vmin=0, vmax=100)
axes_e[0,0].set_title('Observed GHSL (2018)', fontweight='bold', fontsize=12)
im2 = imshow_aspect(axes_e[0,1], ghsl_pred, cmap=ghsl_cmap, vmin=0, vmax=100)
axes_e[0,1].set_title(f'Predicted GHSL', fontweight='bold', fontsize=12)
im3 = imshow_aspect(axes_e[1,0], ghsl_diff, cmap='RdBu_r', vmin=-30, vmax=30)
axes_e[1,0].set_title('Difference (Predicted - Observed)', fontweight='bold', fontsize=12)
axes_e[1,1].axis('off')

fig.colorbar(im1, ax=axes_e[0,:], orientation='vertical', fraction=0.025, pad=0.02, label='GHSL (%)')
fig.colorbar(im3, ax=axes_e[1,:], orientation='vertical', fraction=0.025, pad=0.02, label='Pred - Obs (%)')
plt.savefig(f'{OUTPUT_DIR}/figE_raster_comparison_1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure E saved.')

---
## Cell 7 · Summary

In [ ]:
print('=' * 70)
print(f'  {SOURCE} Milan {PREDICTOR}: random forest tuned in Earth Engine, all {len(gdf_all)} points')
print('=' * 70)
print(f"  chosen: block {best['block']} m, {gm.ee_kwargs(best)}")
print(f"  spatial CV (out-of-fold): RMSE {cv['rmse']:.2f}  MAE {cv['mae']:.2f}  "
      f"R2 {cv['r2']:.3f}  bias {cv['bias']:+.2f}")
print(f'  model record: {MODEL_PATH}')
print(f'  map:          {RF_RASTER_PATH}')